# Feast Feature Store, Step by Step — Iris Dataset

This notebook walks through building a complete Feast feature store from scratch, one small step at a time. Every step is explained in plain language before any code runs, so if this is your first time seeing a feature store, you should be able to follow the whole thing without getting lost.

We use the classic **Iris** flower dataset (150 flower measurements, 3 species) because it is small, fast, and already familiar from earlier sessions — that way the only new thing you're learning here is Feast itself, not the dataset.

By the end of this notebook you will have:

- A real, working Feast feature repository on your own machine
- An **offline store** (used for training) and an **online store** (used for live predictions)
- A trained model that gets its features from the offline store
- A simulated "live" prediction that gets its features from the online store
- A simple check confirming both stores agree with each other

## Before We Start: What Is Feast?

**Feast** (Feature Store) is a free, open-source system for managing machine learning features. Its one job is simple to state but important:

> Make sure a model gets the **exact same feature values**, computed the **exact same way**, whether it's being trained on historical data or making a live prediction right now.

### Why this is harder than it sounds

Without a tool like Feast, teams often end up with feature logic written **twice**: once in a training script (often Python/pandas) and once in a live-serving system (sometimes a completely different language). Even a tiny difference between the two — a different rounding rule, a different time window, a missing row — causes what's called **training-serving skew**: the model was trained on one version of "reality" and is now being asked to make decisions using a slightly different version. This is one of the most common, and most silent, causes of a model quietly getting worse after launch.

Feast's fix is straightforward: define each feature's logic **once**, and give both the training path and the serving path a way to read from that single definition.

### Feast's Core Components

You will meet each of these as a real Python object or a real file later in this notebook. This table is just so the names aren't a surprise when they show up.

| Component | Plain-language definition |
|---|---|
| **Entity** | The "thing" your features are about — a customer, a flower sample, a transaction. Every feature is looked up by an entity's ID. |
| **Data Source** | Where the raw feature data physically lives (a file, a warehouse table, etc.). Feast reads from it — Feast does not store your raw data itself. |
| **Feature View** | A named, versioned group of features that belong to one entity and come from one data source, with a **TTL** (time-to-live: how long a value is considered valid). |
| **Feature Service** | An optional named bundle of feature views used together by one specific model. Not used in this small example, but common once a project has many feature views. |
| **Offline Store** | The batch-oriented storage used for training and historical analysis. In this notebook: a Parquet file, standing in for a real data warehouse. |
| **Online Store** | The low-latency storage used for live, real-time serving. In this notebook: SQLite, standing in for a production system like Redis or DynamoDB. |
| **Registry** | A small metadata database (`registry.db`) that stores all your Entity and Feature View definitions once you "apply" them — this is what makes definitions official and shareable across a team. |
| **Provider** | The environment Feast is configured to run in. We use `local` (everything on your own machine); production setups might say AWS, GCP, or Snowflake instead. |
| **Materialization** | The process that copies the latest feature values from the offline store into the online store, so live serving always has reasonably fresh data. |

### How the pieces fit together

```
DEFINE:   Data Source  --->  Feature View  --->  Registry (apply)
                                                      |
                                     -----------------+-----------------
                                     |                                 |
TRAIN:     Registry ---> Offline Store ---> Historical Features ---> Model Training
                                     |
SERVE:                               ---> Materialization ---> Online Store ---> Live Features ---> Prediction
```

Everything below this point is building exactly this diagram, one box at a time.

## Step 1: Install Feast

Feast is a regular Python package, installed with pip like any other library. If you're running this notebook in a fresh environment, this is the only setup command you need before writing any Feast code.

In [1]:
# Install Feast (and the other libraries we'll use) -- safe to re-run
%pip install feast scikit-learn pandas --quiet

Note: you may need to restart the kernel to use updated packages.


**Checkpoint:** run the cell below. If it prints a version number with no errors, Feast is installed correctly and we can move on.

In [1]:
import feast
print("Feast version:", feast.__version__)

Feast version: 0.64.0


## Step 2: Initialize a Feast Project

A "Feast project" is really just a folder with a specific structure that Feast expects:

```
feature_repo/
  feature_store.yaml   <- configuration (which store types, where the registry lives)
  data/                <- where our offline Parquet file and online SQLite file will live
```

Normally you'd create this by running `feast init my_project` from a terminal, which generates this folder plus some example files. In this notebook we create the same folder structure by hand with plain Python, so nothing is hidden inside a command you haven't seen yet — by the end of this step you'll know exactly what `feast init` would have given you.

In [3]:
import os

REPO_PATH = "feature_repo"
os.makedirs(f"{REPO_PATH}/data", exist_ok=True)

print("Created Feast project folder at:", os.path.abspath(REPO_PATH))
print("Contents so far:", os.listdir(REPO_PATH))

Created Feast project folder at: C:\Shridhar\Study\BITS\Week 9\Demo\feature_repo
Contents so far: ['data']


## Step 3: Prepare the Raw Dataset

Before we can define any features, we need raw data to define features *from*. We'll use scikit-learn's built-in Iris dataset: 150 flower samples, 4 physical measurements each (sepal length/width, petal length/width), and a `target` label (which of 3 species the flower is).

This step has nothing to do with Feast yet — it's the same first step you'd take for any ML project.

In [5]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
raw_df = iris.frame  # a plain pandas DataFrame
print("Shape:", raw_df.shape)
raw_df.head()

Shape: (150, 5)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## Step 4: Perform Feature Engineering

"Feature engineering" means turning raw measurements into the actual inputs a model will use. Here we do two small things:

1. **Rename columns** to valid Python/Feast identifiers. Feast field names can't contain spaces or parentheses, so `"sepal length (cm)"` becomes `sepal_length`.
2. **Engineer one new feature**: `petal_area`, computed as `petal_length * petal_width`. This is a simple but genuinely useful engineered feature — the *area* of the petal often separates species better than length or width alone.

This is exactly the kind of logic that, in a real system, you'd want defined **once** and reused everywhere — which is precisely the problem Feast exists to solve.

In [6]:
df = raw_df.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width",
})

# Engineered feature: petal area
df["petal_area"] = df["petal_length"] * df["petal_width"]

# Human-readable species name, just for our own reference later
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))

df.head()

,sepal_length,sepal_width,petal_length,petal_width,target,petal_area,species
0,5.1,3.5,1.4,0.2,0,0.28,setosa
1,4.9,3.0,1.4,0.2,0,0.28,setosa
2,4.7,3.2,1.3,0.2,0,0.26,setosa
3,4.6,3.1,1.5,0.2,0,0.30,setosa
4,5.0,3.6,1.4,0.2,0,0.28,setosa


## Step 5: Add an Entity ID and Event Timestamp

Feast needs two things on every single row that the raw Iris dataset doesn't naturally have:

- An **entity ID**: something that identifies *which* thing this row of features belongs to. We use the row number itself, calling it `iris_sample_id`.
- An **event timestamp**: *when* this feature value became true. Feast uses this to answer "what did this feature look like at this point in time?" — which is what prevents accidentally using future information during training (a mistake called **data leakage**).

The Iris dataset was collected all at once, so it has no real timestamps. We invent reasonable ones — pretending each sample was recorded 10 minutes after the previous one — purely so Feast has something to work with. In a real project, this timestamp would be genuine: the moment the transaction happened, the moment the sensor reading was taken, etc.

In [9]:
from datetime import datetime, timedelta

df["iris_sample_id"] = df.index

start_time = datetime.now() - timedelta(days=30)
df["event_timestamp"] = [start_time + timedelta(minutes=10 * i) for i in range(len(df))]

# Feast also wants to know when the row was WRITTEN (as opposed to when the
# event happened). In our case they're the same moment, which is normal for
# a simple batch load like this one.
df["created"] = df["event_timestamp"]

df[["iris_sample_id", "event_timestamp", "sepal_length", "petal_area"]].head()

,iris_sample_id,event_timestamp,sepal_length,petal_area
0,0,2026-06-15 21:44:43.763264,5.1,0.28
1,1,2026-06-15 21:54:43.763264,4.9,0.28
2,2,2026-06-15 22:04:43.763264,4.7,0.26
3,3,2026-06-15 22:14:43.763264,4.6,0.30
4,4,2026-06-15 22:24:43.763264,5.0,0.28


## Step 6: Store the Feature Data

Feast's offline store needs to read our engineered features from somewhere physical. We save our DataFrame as a **Parquet file** — a column-oriented file format that's fast to read and is the standard choice for feature stores (the same role a table in a data warehouse like BigQuery or Snowflake would play in a larger production system).

This Parquet file is, from this point on, the "single source of truth" for our features.

In [11]:
feature_data_path = os.path.abspath(f"{REPO_PATH}/data/iris_features.parquet")
df.to_parquet(feature_data_path, index=False)

print("Saved feature data to:", feature_data_path)

Saved feature data to: C:\Shridhar\Study\BITS\Week 9\Demo\feature_repo\data\iris_features.parquet


## Step 7: Configure `feature_store.yaml`

Every Feast project has one configuration file, `feature_store.yaml`, that tells Feast three things:

- `project`: a name for this feature store (keeps it separate from other Feast projects on the same machine)
- `provider`: `local` means everything runs on this machine with no cloud services involved
- `registry`: the file path where Feast will keep track of every Entity and Feature View we define
- `online_store`: which technology serves low-latency lookups — we use `sqlite`, a simple file-based database, standing in for Redis/DynamoDB in a real deployment

We write this file directly from Python so you can see its exact contents, but in a normal project you'd typically hand-edit this file once and rarely touch it again.

In [13]:
feature_store_yaml = """project: iris_beginner_demo
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
"""

with open(f"{REPO_PATH}/feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print(feature_store_yaml)

project: iris_beginner_demo
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3



## Step 8: Define the Entity

Recall from the components table: an **Entity** is the "thing" our features describe. In this project, that's a single flower sample, identified by `iris_sample_id`.

Every feature we define later will be looked up by this entity — "give me the features for sample #42" is a question Feast can only answer because we've told it that `iris_sample_id` is a valid way to identify a row.

In [15]:
from feast import Entity

iris_sample = Entity(
    name="iris_sample_id",
    description="A single iris flower measurement sample",
)

print("Entity defined:", iris_sample.name)

Entity defined: iris_sample_id


C:\Users\galan\AppData\Local\Temp\ipykernel_5392\1269629037.py:3: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'iris_sample_id'.
  iris_sample = Entity(


## Step 9: Define the Data Source

Now we tell Feast exactly where to find our raw feature data on disk: the Parquet file we saved in Step 6. We also tell it which column is the event timestamp and which is the "created" timestamp, so Feast knows how to reason about time for this data.

In [17]:
from feast import FileSource

iris_source = FileSource(
    path=feature_data_path,
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

print("Data source points to:", iris_source.path)

Data source points to: C:\Shridhar\Study\BITS\Week 9\Demo\feature_repo\data\iris_features.parquet


## Step 10: Define the Feature View

The **Feature View** ties everything together: it says "these specific features, for this entity, come from this data source, and stay valid for this long (TTL)."

We register five features: the four original measurements plus our engineered `petal_area`. Notice we do **not** include `target` or `species` here — those are labels, not features, and a feature store is for features that a model consumes as input, not the answer we're trying to predict.

In [19]:
from datetime import timedelta
from feast import FeatureView, Field
from feast.types import Float32

iris_feature_view = FeatureView(
    name="iris_features",
    entities=[iris_sample],
    ttl=timedelta(days=365),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="petal_area", dtype=Float32),
    ],
    source=iris_source,
    online=True,
)

print("Feature view defined:", iris_feature_view.name)
print("Features in this view:", [f.name for f in iris_feature_view.features])

Feature view defined: iris_features
Features in this view: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'petal_area']


## Step 11: Run `feast apply` to Register the Definitions

So far we've only created Python *objects* describing our Entity and Feature View — Feast doesn't know about them yet. **Applying** them writes their definitions into the registry (`registry.db`), which is what makes them official and query-able.

From a terminal, this step is normally the command `feast apply`. Here, we call the exact same operation through Feast's Python SDK with `store.apply(...)`, which does the same thing without leaving the notebook.

In [21]:
from feast import FeatureStore

store = FeatureStore(repo_path=REPO_PATH)
store.apply([iris_sample, iris_feature_view])

print("Registered feature views:")
for fv in store.list_feature_views():
    print(" -", fv.name, "with features:", [f.name for f in fv.features])

Registered feature views:
 - iris_features with features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'petal_area']


## Step 12: Retrieve Historical Features for Training

Now for the payoff of all that setup: asking Feast for training data.

We give Feast an **entity dataframe** — just the entity IDs and timestamps we care about (plus, conveniently, our label column `target` along for the ride) — and Feast returns the feature values that were correct **as of each of those timestamps**. This is called a **point-in-time join**, and it's what guarantees no row accidentally uses information from the future — the exact mistake that causes data leakage.

In [23]:
entity_df = df[["iris_sample_id", "event_timestamp", "target"]]

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
        "iris_features:petal_area",
    ],
).to_df()

print("Training dataframe shape:", training_df.shape)
training_df.head()

Training dataframe shape: (150, 8)


,iris_sample_id,event_timestamp,target,sepal_length,sepal_width,petal_length,petal_width,petal_area
0,0,2026-06-15 21:44:43.763264+00:00,0,5.1,3.5,1.4,0.2,0.28
1,1,2026-06-15 21:54:43.763264+00:00,0,4.9,3.0,1.4,0.2,0.28
2,2,2026-06-15 22:04:43.763264+00:00,0,4.7,3.2,1.3,0.2,0.26
3,3,2026-06-15 22:14:43.763264+00:00,0,4.6,3.1,1.5,0.2,0.30
4,4,2026-06-15 22:24:43.763264+00:00,0,5.0,3.6,1.4,0.2,0.28


## Step 13: Create the Training Dataset

With the feature values in hand, this step is standard machine learning data preparation: separate features (`X`) from the label (`y`), and split into a training set and a held-out test set so we can honestly evaluate the model afterward.

In [25]:
from sklearn.model_selection import train_test_split

feature_columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "petal_area"]

X = training_df[feature_columns]
y = training_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

Training rows: 120
Test rows: 30


## Step 14: Train the Machine Learning Model

We train a simple `RandomForestClassifier`. The model itself is intentionally not the focus of this notebook — any classifier would demonstrate the same point. What matters is that its inputs came from our Feast **offline store**, via the point-in-time-correct retrieval in Step 12.

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)

test_predictions = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Test accuracy: {test_accuracy:.3f}")

Test accuracy: 0.967


## Step 15: Materialize Features to the Online Store

Our model is trained, but right now our online store is empty — nothing has been served to it yet. **Materialization** is the step that copies the latest feature values from the offline Parquet file into the online SQLite store, exactly as a scheduled job would do on a recurring basis in production (e.g. every few minutes or every hour, depending on how fresh features need to be).

After this step, the online store can answer "what are this entity's features **right now**?" with a fast lookup, instead of scanning the whole Parquet file.

In [29]:
from datetime import datetime

store.materialize_incremental(end_date=datetime.now())
print("Materialization complete -- online store is now populated.")

Materializing 1 feature views to 2026-07-15 21:45:07+00:00 into the sqlite online store.

iris_features from 2025-07-15 16:15:07+00:00 to 2026-07-15 21:45:07+00:00:
Materialization complete -- online store is now populated.


## Step 16: Retrieve Online Features for Inference

Now we simulate what a real production system does at prediction time: given just an entity ID (no timestamp needed this time — online lookups always return the *current* value), fetch that entity's features from the low-latency online store.

We'll do this for a handful of sample IDs, as if five separate prediction requests just arrived.

In [31]:
sample_ids = df["iris_sample_id"].sample(5, random_state=7).tolist()

online_features = store.get_online_features(
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
        "iris_features:petal_area",
    ],
    entity_rows=[{"iris_sample_id": sid} for sid in sample_ids],
).to_df()

online_features

,iris_sample_id,sepal_width,petal_area,petal_width,petal_length,sepal_length
0,149,3.0,9.18,1.8,5.1,5.9
1,84,3.0,6.75,1.5,4.5,5.4
2,40,3.5,0.39,0.3,1.3,5.0
3,66,3.0,6.75,1.5,4.5,5.6
4,106,2.5,7.65,1.7,4.5,4.9


## Step 17: Pass the Features to the ML Model, and Generate a Prediction

Two small things matter here, both of which are common real-world sources of bugs:

1. The columns must be in the **same order and with the same names** the model was trained on. `online_features` already has the right column names because we asked for the same feature names in Step 16 — but it's worth noticing that this alignment is not automatic in every system, which is exactly the kind of subtle mismatch that causes training-serving skew if nobody checks it.
2. We select only the feature columns the model expects (`feature_columns`), dropping the entity ID column before calling `.predict()`.

In [33]:
X_serve = online_features[feature_columns]
predicted_class = model.predict(X_serve)

species_lookup = dict(enumerate(iris.target_names))
result = online_features[["iris_sample_id"]].copy()
result["predicted_species"] = [species_lookup[c] for c in predicted_class]

result

,iris_sample_id,predicted_species
0,149,virginica
1,84,versicolor
2,40,setosa
3,66,versicolor
4,106,virginica


### Generate the Prediction — Compare Against Ground Truth

Since this is a demo, we happen to know the real species for these samples. In a genuine production system you wouldn't have this luxury at prediction time — but checking it here is a good sanity test that the whole pipeline, end to end, is behaving correctly.

In [35]:
true_species = df[df["iris_sample_id"].isin(sample_ids)][["iris_sample_id", "species"]]

result = result.merge(true_species, on="iris_sample_id")
result["correct"] = result["predicted_species"] == result["species"]

result

,iris_sample_id,predicted_species,species,correct
0,149,virginica,virginica,True
1,84,versicolor,versicolor,True
2,40,setosa,setosa,True
3,66,versicolor,versicolor,True
4,106,virginica,virginica,True


## Step 18: Monitor and Manage Feature Freshness and Consistency

The last piece of a healthy feature store isn't a one-time step — it's an ongoing habit. Two things are worth checking continuously in any real deployment:

**Consistency**: do the offline and online stores agree on the same entity's feature values? If they don't, that's training-serving skew happening in real time. Let's check it directly, for the same five samples we just served.

**Freshness**: how old is the data sitting in the online store? If materialization stops running (a failed cron job, a broken pipeline), the online store will keep serving increasingly stale values without any obvious error — it will simply keep answering, just with old data.

In [37]:
# --- Consistency check: offline vs online, same entities ---
offline_values = df[df["iris_sample_id"].isin(sample_ids)][
    ["iris_sample_id", "petal_area"]
].rename(columns={"petal_area": "petal_area_offline"})

online_values = online_features[["iris_sample_id", "petal_area"]].rename(
    columns={"petal_area": "petal_area_online"}
)

consistency_check = online_values.merge(offline_values, on="iris_sample_id")
consistency_check["difference"] = (
    consistency_check["petal_area_online"] - consistency_check["petal_area_offline"]
).abs()

consistency_check

,iris_sample_id,petal_area_online,petal_area_offline,difference
0,149,9.18,9.18,3.051758e-07
1,84,6.75,6.75,0.000000e+00
2,40,0.39,0.39,1.430511e-08
3,66,6.75,6.75,0.000000e+00
4,106,7.65,7.65,9.536743e-08


In [39]:
# --- Freshness check: how old is the data in the online store? ---
most_recent_event = df["event_timestamp"].max()
now = datetime.now()
staleness = now - most_recent_event.to_pydatetime().replace(tzinfo=None)

print("Most recent feature timestamp:", most_recent_event)
print("Current time:                ", now)
print("Staleness:                   ", staleness)
print()
print("In production, this staleness number is exactly what you would feed into")
print("a monitoring dashboard or alert: e.g. 'page someone if staleness exceeds 1 hour.'")

Most recent feature timestamp: 2026-06-16 22:34:43.763264
Current time:                 2026-07-15 21:45:19.424480
Staleness:                    28 days, 23:10:35.661216

In production, this staleness number is exactly what you would feed into
a monitoring dashboard or alert: e.g. 'page someone if staleness exceeds 1 hour.'


If `difference` is at (or extremely close to) zero for every row, the two stores agree — the feature store is doing its job. If materialization had failed, or if the online-serving code path had been written to compute `petal_area` slightly differently than the offline path, this exact check is what would have caught it — automatically, and long before a person noticed the model getting worse.

## Wrap-Up

Starting from nothing, we built:

1. A real Feast project on disk, with a registry, an offline Parquet store, and an online SQLite store
2. One shared feature definition (`iris_features`) used for both training and serving
3. A model trained on point-in-time-correct historical features
4. A materialization job that kept the online store in sync with the offline store
5. A simulated live prediction, served entirely from the online store
6. A consistency and freshness check — the habit that catches skew before it becomes a production incident

Every one of Feast's core components from the introduction — Entity, Data Source, Feature View, Registry, Offline Store, Online Store, Materialization — showed up as a real, working piece of this notebook. The dataset was small and the model was simple on purpose: the pattern you just built scales to real production systems with millions of rows and dozens of features without changing shape at all.